# Project 2 Error analysis of the selected model

Use this notebook with the application-only 64 → 32 model in `gil_project2.ipynb`. Default inputs are the original saved seed-42 baseline (threshold 0.139), preserving our ten-error review. A new main-model export can be selected explicitly below; it receives a separate review folder.

Only predictions and `application_train.csv` are needed. The code does not retrain, load bureau/payment tables, or require another notebook. Individual inspection uses a fixed 500-applicant validation eyeball pool. The remaining validation rows are used only for aggregate checks. Earlier tuning used the full validation set; partitioning it now does not make it untouched.

The CSV records observations, hypotheses, checks, and decisions. Leave human-review fields blank until reviewed. True negatives and true positives are comparison cases, not errors. The final conclusions distinguish evidence from possible explanations.

## 1. Select the saved model run

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

folders = [Path.cwd(), Path('/content'), Path.home() / 'Downloads' / 'Project 2 DATA']
data_path = next((folder / 'application_train.csv' for folder in folders
                  if (folder / 'application_train.csv').exists()), None)
# Keep None to use the original baseline; set Path('model_runs/application_...')
# to the folder printed by gil_project2.ipynb for a new model run.
INPUT_DIR = None
if INPUT_DIR is None:
    input_folders = [folder / 'error_analysis_inputs' for folder in folders] + folders
    input_dir = next((folder for folder in input_folders
                      if (folder / 'validation_predictions.json').exists()
                      and (folder / 'model_info.json').exists()), None)
else:
    input_dir = Path(INPUT_DIR)
if data_path is None or input_dir is None:
    raise FileNotFoundError('Provide application_train.csv, validation_predictions.json, and model_info.json.')

df = pd.read_csv(data_path)
predictions = pd.read_json(input_dir / 'validation_predictions.json')
model_info = json.loads((input_dir / 'model_info.json').read_text())
threshold = model_info['threshold']
print(model_info)
print('Saved validation applicants:', len(predictions))

{'source_run': 'current_model_20261001T033358362869Z', 'model': 'numeric missingness + employment placeholder fix', 'training_seed': 42, 'threshold': 0.139, 'validation_auc': 0.7478152451243464, 'selected_epoch': 5, 'split': '60/20/20 stratified, seed 42', 'note': 'Saved CPU baseline, not automatically updated when the main notebook is retrained.'}
Saved validation applicants: 61502


## 2. Match predictions to applicants and label errors
A false negative means actual difficulty but predicted no difficulty. A false positive means predicted difficulty but actual no difficulty. The original TARGET describes repayment difficulty, not necessarily permanent failure to repay.

In [2]:
assert df['SK_ID_CURR'].is_unique and predictions['SK_ID_CURR'].is_unique
assert predictions['predicted_probability'].between(0, 1).all()
feature_columns = ['SK_ID_CURR', 'TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
                   'DAYS_BIRTH', 'CODE_GENDER', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
review = predictions.merge(df[feature_columns].copy(), on='SK_ID_CURR', how='left', validate='one_to_one', indicator=True).copy()
assert review['_merge'].eq('both').all(), 'Some applicant IDs were not found.'
assert review['actual_TARGET'].eq(review['TARGET']).all(), 'Targets do not match the saved predictions.'
expected_labels = (review['predicted_probability'] >= threshold).astype(int)
assert review['predicted_label'].eq(expected_labels).all(), 'Saved labels do not match the threshold.'

actual = review['actual_TARGET']
predicted = review['predicted_label']
review['outcome'] = np.select([
    (actual == 1) & (predicted == 0),
    (actual == 0) & (predicted == 1),
    (actual == 1) & (predicted == 1)
], ['False negative', 'False positive', 'True positive'], default='True negative')
review['age_years'] = (-review['DAYS_BIRTH'] / 365.25).round(1)
review['employment_placeholder'] = review['DAYS_EMPLOYED'].eq(365243).astype(int)
review['threshold'] = threshold
review['model_run'] = model_info['source_run']
review['distance_from_threshold'] = review['predicted_probability'] - threshold

display(pd.DataFrame(confusion_matrix(actual, predicted, labels=[0, 1]),
    index=['Actual no difficulty', 'Actual difficulty'],
    columns=['Predicted no difficulty', 'Predicted difficulty']))
display(pd.Series({
    'AUC': roc_auc_score(actual, review['predicted_probability']),
    'Accuracy': accuracy_score(actual, predicted),
    'Precision': precision_score(actual, predicted, zero_division=0),
    'Recall': recall_score(actual, predicted, zero_division=0),
    'F1': f1_score(actual, predicted, zero_division=0)
}, name='Full validation metrics').round(4))

                      Predicted no difficulty  Predicted difficulty
Actual no difficulty                    48767                  7770
Actual difficulty                        2765                  2200
AUC          0.7478
Accuracy     0.8287
Precision    0.2207
Recall       0.4431
F1           0.2946
Name: Full validation metrics, dtype: float64


## 3. Reserve an eyeball review pool
Use 500 reproducibly chosen validation applicants for manual inspection. Keep the remainder as a blackbox validation subset for aggregate evaluation. This split is stratified on actual TARGET, not on whether the model made an error.

**Important:** the full validation set already informed earlier model selection and threshold tuning. This new partition organizes future review; it does not undo that exposure or create an untouched holdout. Do not examine individual blackbox rows for manual review.

In [3]:
REVIEW_SEED = 42
EYEBALL_SIZE = 500
eyeball, blackbox = train_test_split(
    review, train_size=EYEBALL_SIZE, random_state=REVIEW_SEED,
    stratify=review['actual_TARGET']
)
assert set(eyeball['SK_ID_CURR']).isdisjoint(blackbox['SK_ID_CURR'])
print('Eyeball applicants:', len(eyeball), '| Blackbox applicants:', len(blackbox))
print('Available cases in the eyeball pool:')
display(eyeball['outcome'].value_counts())

Eyeball applicants: 500 | Blackbox applicants: 61002
Available cases in the eyeball pool:
outcome
True negative     386
False positive     74
False negative     26
True positive      14
Name: count, dtype: int64


## 4. Select the first ten errors

In [4]:
CASES_PER_TYPE = 5
samples = []
for error_type in ['False negative', 'False positive']:
    cases = eyeball[eyeball['outcome'] == error_type].sort_values('SK_ID_CURR')
    samples.append(cases.sample(n=min(CASES_PER_TYPE, len(cases)), random_state=REVIEW_SEED))
sample = pd.concat(samples, ignore_index=True)

columns = [
    'SK_ID_CURR', 'outcome', 'actual_TARGET', 'predicted_label',
    'predicted_probability', 'threshold', 'distance_from_threshold',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'age_years', 'CODE_GENDER', 'DAYS_EMPLOYED', 'employment_placeholder',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'model_run'
]
sample = sample[columns].copy()
for column in ['reviewer', 'observation', 'suspected_pattern', 'proposed_check', 'proposed_change', 'result_after_change']:
    sample[column] = ''
display(sample[['SK_ID_CURR', 'outcome', 'predicted_probability', 'AMT_INCOME_TOTAL', 'AMT_CREDIT']])

   SK_ID_CURR         outcome  ...  AMT_INCOME_TOTAL  AMT_CREDIT
0      215363  False negative  ...           90000.0    518562.0
1      276559  False negative  ...          101250.0     99000.0
2      102959  False negative  ...          333000.0    297130.5
3      436136  False negative  ...          112500.0    654498.0
4      222610  False negative  ...          112500.0    436032.0
5      121564  False positive  ...          202500.0   1078200.0
6      405943  False positive  ...          153000.0    270000.0
7      209864  False positive  ...           58500.0    272520.0
8      102331  False positive  ...           54000.0    180000.0
9      283710  False positive  ...          157500.0    557770.5

[10 rows x 5 columns]


## 5. Save exactly ten errors

The review CSV contains five false negatives and five false positives from the selected application-only model. Existing notes for these applicants are preserved. If an older CSV includes extra cases, it is backed up before the file is reduced. Correct cases appear only in supporting comparisons, not as extra rows in the ten-error review CSV.


In [5]:
import shutil
from datetime import datetime, timezone
run_name = model_info['source_run']
assert Path(run_name).name == run_name, 'Invalid run identifier'
output_dir = Path('error_analysis') if run_name == 'current_model_20261001T033358362869Z' else Path('error_analysis') / run_name
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'starter_review_seed42.csv'
assert len(sample) == 10 and sample['outcome'].value_counts().to_dict() == {'False negative': 5, 'False positive': 5}
selected_ids = sample.SK_ID_CURR.astype(str).tolist()
if output_path.exists():
    previous = pd.read_csv(output_path, dtype=str, keep_default_na=False)
    assert previous.SK_ID_CURR.is_unique
    assert previous.model_run.eq(run_name).all(), 'Review belongs to another model.'
    assert np.allclose(previous.threshold.astype(float), threshold)
    selected = previous.set_index('SK_ID_CURR').reindex(selected_ids)
    # Retain every existing field/note for selected IDs; fill genuinely new IDs.
    for applicant_id in selected_ids:
        if applicant_id not in set(previous.SK_ID_CURR):
            fresh = sample[sample.SK_ID_CURR.astype(str).eq(applicant_id)].iloc[0]
            for field in fresh.index:
                if field != 'SK_ID_CURR': selected.loc[applicant_id, field] = fresh[field]
    selected = selected.reset_index().fillna('')
    if previous.SK_ID_CURR.tolist() != selected_ids:
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        backup = output_path.with_name(output_path.stem + '_before_ten_' + stamp + '.csv')
        shutil.copy2(output_path, backup)
        selected.to_csv(output_path, index=False)
        print('Preserved previous review:', backup.resolve())
else:
    sample.to_csv(output_path, index=False)
manifest_path = output_dir / 'review_selection_seed42.json'
if not manifest_path.exists():
    manifest_path.write_text(json.dumps({
        'model_info': model_info, 'review_seed': REVIEW_SEED,
        'eyeball_ids': sorted(eyeball.SK_ID_CURR.tolist()),
        'sample_ids': sample.SK_ID_CURR.tolist(), 'blackbox_count': len(blackbox)
    }, indent=2))
review_csv = pd.read_csv(output_path, keep_default_na=False)
assert len(review_csv) == 10 and review_csv.SK_ID_CURR.is_unique
print('Ten-error review:', output_path.resolve())
display(review_csv[['SK_ID_CURR', 'outcome', 'predicted_probability']])


Preserved previous review: /Users/gillazar/Downloads/Project 2 DATA/error_analysis/starter_review_seed42_before_ten_20261007T171639706081Z.csv
Ten-error review: /Users/gillazar/Downloads/Project 2 DATA/error_analysis/starter_review_seed42.csv
   SK_ID_CURR         outcome  predicted_probability
0      215363  False negative               0.071915
1      276559  False negative               0.081685
2      102959  False negative               0.126614
3      436136  False negative               0.074941
4      222610  False negative               0.110617
5      121564  False positive               0.204768
6      405943  False positive               0.243140
7      209864  False positive               0.398838
8      102331  False positive               0.157707
9      283710  False positive               0.237037


## 6. Compare errors with correct cases

In [6]:
external_scores = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
missing_rates = eyeball[external_scores].isna().groupby(eyeball['outcome']).mean()
missing_rates.insert(0, 'applicants', eyeball.groupby('outcome').size())
missing_rates.round(3)

                applicants  EXT_SOURCE_1  EXT_SOURCE_2  EXT_SOURCE_3
outcome                                                             
False negative          26         0.538         0.000         0.115
False positive          74         0.554         0.014         0.257
True negative          386         0.565         0.003         0.199
True positive           14         0.571         0.000         0.214


## 7. Compare sampled errors with similar correct cases
For each false positive, choose the true negative in the eyeball pool with the closest EXT_SOURCE_2. For each false negative, choose the closest true positive. Require a score difference no greater than 0.05; otherwise mark unmatched. Missing scores cannot be matched. Ties use applicant ID. Matching is with replacement, so a correct applicant may be reused. This is an exploratory comparison on one variable, not a causal comparison or independent matched trial. Other applicant characteristics can differ.

In [7]:
paired_rows = []
for _, error in sample.iterrows():
    target_type = 'True negative' if error['outcome'] == 'False positive' else 'True positive'
    controls = eyeball[eyeball['outcome'].eq(target_type) & eyeball['EXT_SOURCE_2'].notna()].copy()
    controls['score_gap'] = (controls['EXT_SOURCE_2'] - error['EXT_SOURCE_2']).abs()
    controls = controls.sort_values(['score_gap', 'SK_ID_CURR'])
    matched = (pd.notna(error['EXT_SOURCE_2']) and len(controls) > 0
               and controls.iloc[0]['score_gap'] <= 0.05)
    row = {'error_id': int(error['SK_ID_CURR']), 'error_type': error['outcome'],
           'match_status': 'Matched' if matched else 'No match within 0.05',
           'control_id': None, 'control_type': target_type, 'score_gap': None}
    for field in ['predicted_probability', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
                  'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'age_years']:
        row['error_' + field] = error[field]
        row['control_' + field] = controls.iloc[0][field] if matched else None
    if matched:
        row['control_id'] = int(controls.iloc[0]['SK_ID_CURR'])
        row['score_gap'] = float(controls.iloc[0]['score_gap'])
    paired_rows.append(row)
pairs = pd.DataFrame(paired_rows)
display(pairs[['error_id', 'error_type', 'match_status', 'control_id', 'score_gap',
               'error_EXT_SOURCE_3', 'control_EXT_SOURCE_3']].round(4))
print('Distinct correct comparison applicants:', pairs['control_id'].nunique())

   error_id      error_type  ... error_EXT_SOURCE_3  control_EXT_SOURCE_3
0    215363  False negative  ...             0.1885                   NaN
1    276559  False negative  ...             0.4597                0.3186
2    102959  False negative  ...             0.2721                0.3690
3    436136  False negative  ...             0.5209                0.3186
4    222610  False negative  ...             0.3377                0.3690
5    121564  False positive  ...             0.2472                0.6058
6    405943  False positive  ...             0.1777                0.4939
7    209864  False positive  ...             0.4454                0.6578
8    102331  False positive  ...             0.3842                0.8106
9    283710  False positive  ...             0.3945                0.4939

[10 rows x 7 columns]
Distinct correct comparison applicants: 7


## 8. Check the external score pattern in the eyeball pool
Use broad descriptive bands rather than searching for a favorable cutoff. We examine all actual negatives in the eyeball pool with EXT_SOURCE_2 at or below 0.25, including correct negatives. Counts and missing values matter: a difference in medians alone does not establish significance or causation.

In [8]:
low_score_negatives = eyeball[(eyeball['actual_TARGET'] == 0) & (eyeball['EXT_SOURCE_2'] <= 0.25)]
score_comparison = low_score_negatives.groupby('outcome').agg(
    applicants=('SK_ID_CURR', 'size'),
    external3_available=('EXT_SOURCE_3', 'count'),
    median_external3=('EXT_SOURCE_3', 'median'),
    median_age=('age_years', 'median'),
    median_income=('AMT_INCOME_TOTAL', 'median'))
display(score_comparison.round(4))

                applicants  external3_available  ...  median_age  median_income
outcome                                          ...                           
False positive          35                   23  ...        35.5       142587.0
True negative           31                   29  ...        43.0       135000.0

[2 rows x 5 columns]


## 9. Check age patterns across the eyeball pool

Compare all eligible cases in the review pool, not only the chosen rows. Medians exclude missing values; counts of available scores are displayed. These descriptive comparisons do not isolate a feature's causal effect.

In [9]:
age_check = eyeball.copy()
age_check['exact_age'] = -age_check.DAYS_BIRTH / 365.25
age_check['credit_income'] = age_check.AMT_CREDIT / age_check.AMT_INCOME_TOTAL.replace(0, np.nan)
for name, subset in [
    ('Age 45+ with actual difficulty', age_check[(age_check.exact_age >= 45) & age_check.actual_TARGET.eq(1)]),
    ('Under 30 without actual difficulty', age_check[(age_check.exact_age < 30) & age_check.actual_TARGET.eq(0)])]:
    print(name)
    display(subset.groupby('outcome').agg(
        applicants=('SK_ID_CURR', 'size'),
        ext2_available=('EXT_SOURCE_2', 'count'), median_ext2=('EXT_SOURCE_2', 'median'),
        ext3_available=('EXT_SOURCE_3', 'count'), median_ext3=('EXT_SOURCE_3', 'median'),
        median_credit_income=('credit_income', 'median')).round(4))

Age 45+ with actual difficulty
                applicants  ext2_available  ...  median_ext3  median_credit_income
outcome                                     ...                                   
False negative          14              14  ...       0.4212                4.4379
True positive            4               4  ...       0.3159                4.5862

[2 rows x 6 columns]
Under 30 without actual difficulty
                applicants  ext2_available  ...  median_ext3  median_credit_income
outcome                                     ...                                   
False positive          24              23  ...       0.2366                2.8301
True negative           41              41  ...       0.4975                2.1980

[2 rows x 6 columns]


## 10. Evaluate fairness at the saved model threshold
Use full-validation aggregate metrics, with no individual blackbox review. Positive means predicted repayment difficulty. The favorable model outcome for this analysis is predicted no difficulty (label 0), not a claim that the prediction alone constitutes a lending decision. Report recorded CODE_GENDER categories as stored; do not infer identities.

False-positive rate is the fraction of actual negatives incorrectly flagged. Recall is the fraction of actual positives flagged. Favorable prediction rate is the fraction predicted 0; true-negative rate is 1 minus false-positive rate. Compare these along with group sizes and actual target rates. Age bands are fixed at under 30, 30 to under 45, 45 to under 60, and 60 or older. Wilson intervals are approximate 95% intervals for FPR and recall, not corrections for repeated comparisons or earlier tuning. Small groups need caution. Group disparities are descriptive, not proof of discrimination or causation.

In [10]:
def wilson_interval(successes, total):
    if total == 0:
        return np.nan, np.nan
    rate = successes / total
    z = 1.96
    denominator = 1 + z**2 / total
    center = (rate + z**2 / (2 * total)) / denominator
    radius = z * np.sqrt(rate * (1-rate) / total + z**2 / (4 * total**2)) / denominator
    return center-radius, center+radius

fairness_data = review.copy()
# Compute bands from exact age, not the rounded age used in review displays.
fairness_data['age_group'] = pd.cut(-fairness_data['DAYS_BIRTH'] / 365.25,
    bins=[0, 30, 45, 60, np.inf], right=False, labels=['Under 30', '30 to under 45', '45 to under 60', '60 and older'])
fairness_rows = []
for attribute in ['CODE_GENDER', 'age_group']:
    for group_name, group in fairness_data.groupby(attribute, observed=True, dropna=False):
        a, pred = group['actual_TARGET'], group['predicted_label']
        tp = int(((a == 1) & (pred == 1)).sum())
        fp = int(((a == 0) & (pred == 1)).sum())
        positives, negatives = int(a.sum()), int((a == 0).sum())
        fpr = fp / negatives if negatives else np.nan
        recall = tp / positives if positives else np.nan
        fpr_low, fpr_high = wilson_interval(fp, negatives)
        recall_low, recall_high = wilson_interval(tp, positives)
        fairness_rows.append(dict(attribute=attribute, group=str(group_name), n=len(group),
            actual_positives=positives, actual_negatives=negatives,
            false_positives=fp, false_negatives=positives-tp,
            positive_rate=float(a.mean()), favorable_prediction_rate=float((pred == 0).mean()),
            false_positive_rate=fpr, true_negative_rate=1-fpr, recall=recall,
            fpr_ci_low=fpr_low, fpr_ci_high=fpr_high,
            recall_ci_low=recall_low, recall_ci_high=recall_high,
            precision=tp/(tp+fp) if tp+fp else np.nan,
            auc=float(roc_auc_score(a, group['predicted_probability'])) if a.nunique() == 2 else np.nan,
            small_group_warning=positives < 20 or negatives < 20))
fairness = pd.DataFrame(fairness_rows)
display(fairness[['attribute', 'group', 'n', 'actual_positives', 'positive_rate',
                 'false_positive_rate', 'recall', 'favorable_prediction_rate', 'small_group_warning']].round(4))

     attribute           group  ...  favorable_prediction_rate  small_group_warning
0  CODE_GENDER               F  ...                     0.8747                False
1  CODE_GENDER               M  ...                     0.7667                False
2  CODE_GENDER             XNA  ...                     1.0000                 True
3    age_group        Under 30  ...                     0.6711                False
4    age_group  30 to under 45  ...                     0.8203                False
5    age_group  45 to under 60  ...                     0.8910                False
6    age_group    60 and older  ...                     0.9580                False

[7 rows x 9 columns]


## 11. Save aggregate analysis alongside the review CSV

In [11]:
from datetime import datetime, timezone
analysis_dir = output_dir / ('analysis_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
analysis_dir.mkdir(parents=True, exist_ok=False)
pairs.to_csv(analysis_dir / 'matched_comparisons.csv', index=False)
fairness.to_csv(analysis_dir / 'fairness_metrics.csv', index=False)
missing_rates.to_csv(analysis_dir / 'missingness_by_outcome.csv')
(analysis_dir / 'model_info.json').write_text(json.dumps(model_info, indent=2))
print('Review CSV:', output_path.resolve())
print('Aggregate analysis:', analysis_dir.resolve())
print('Existing observation and reviewer notes remain unchanged.')

Review CSV: /Users/gillazar/Downloads/Project 2 DATA/error_analysis/starter_review_seed42.csv
Aggregate analysis: /Users/gillazar/Downloads/Project 2 DATA/error_analysis/analysis_20261007T171639755721Z
Existing observation and reviewer notes remain unchanged.


## 12. Evidence and decisions from the completed review

These findings refer to the original saved application-only baseline, not automatically to a newly selected run.

- The initial ten errors suggested an external-score interaction. Testing it produced a tiny, inconsistent AUC gain and slightly lower average F1, so it was not adopted.
- Younger false positives tended to have lower external scores than younger true negatives. Older false negatives tended to have higher external scores than older true positives. These associations identify weaknesses but do not establish excessive feature reliance or individual causes.
- An apparent high loan-to-income pattern in five older missed cases weakened across the full review pool. We did not treat it as established evidence.
- A separate completed inference check reproduced the original saved model and compared training and validation aggregates. Among ages 45+ with actual difficulty and EXT_SOURCE_2 ≥ 0.50, the missed-case rates were 86.1% training and 87.8% validation. Among under-30 actual negatives with EXT_SOURCE_3 < 0.25, false-alarm rates were 65.7% in both. The pattern therefore exists during fitting as well as generalization. This historical check is documented here; running this notebook does not require its helper script or checkpoint files.
- Five architectures across three seeds did not justify changing the original 64 → 32 network. Prolonged training caused overfitting, supporting best-checkpoint selection.
- Bureau and repayment history did add predictive information. The repayment model improved mean validation AUC to 0.75692 and F1 to 0.30348, but added complexity and worsened some false alarms. We chose the application-only model for the current scope, explicitly accepting the performance tradeoff.

### Fairness interpretation

Prioritize false-positive-rate parity to examine incorrectly flagging borrowers without difficulty; track recall alongside it to expose missed difficulties. Equalized odds would require both rates to be similar across groups. TARGET 1 is unfavorable repayment difficulty; predicted 0 is the favorable model outcome, not an actual approval decision. Observed disparities are real descriptive findings, but their cause is not established. A one-person recorded-gender category cannot support a meaningful fairness conclusion. Age and recorded gender remain model inputs; we have not established group unawareness or fairness simply by measuring these rates.

### Complete the review

For each selected case or group comparison, record observable facts, a possible explanation, a check, and the resulting decision. It is acceptable to write that the cause remains unknown. Do not fabricate explanations for every row. Existing experiment notebooks remain historical evidence; they are not required to run these two active notebooks.

This workflow is ready for continued review. Team interpretation, the final project report, and final evaluation are still required. On Colab, upload the application CSV and selected prediction-input folder; download review outputs through the Files sidebar.